>
---
Packages imports

In [32]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

# Load your dataset
df = pd.read_csv('../data/diabetes.csv')



>
---
Data preprocessing & cleaning

In [33]:
# List of columns where 0 is invalid
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Replace 0 with NaN, then fill with Median
for col in cols_to_fix:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print(df)    

     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6    148.0           72.0           35.0    125.0  33.6   
1              1     85.0           66.0           29.0    125.0  26.6   
2              8    183.0           64.0           29.0    125.0  23.3   
3              1     89.0           66.0           23.0     94.0  28.1   
4              0    137.0           40.0           35.0    168.0  43.1   
..           ...      ...            ...            ...      ...   ...   
763           10    101.0           76.0           48.0    180.0  32.9   
764            2    122.0           70.0           27.0    125.0  36.8   
765            5    121.0           72.0           23.0    112.0  26.2   
766            1    126.0           60.0           29.0    125.0  30.1   
767            1     93.0           70.0           31.0    125.0  30.4   

     DiabetesPedigreeFunction  Age  Outcome  
0                       0.627   50        1  
1                  

>
---
Data Scaling & splitting

In [34]:


X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# The data has 99 Healthy cases but only 55 Diabetic cases in the test set.
# APPLY SMOTE ONLY TO TRAINING DATA to fix the imbalance and prevent data leakage.
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Original training shape: {y_train.value_counts()}")
print(f"Balanced training shape: {y_train_balanced.value_counts()}")

Original training shape: Outcome
0    401
1    213
Name: count, dtype: int64
Balanced training shape: Outcome
0    401
1    401
Name: count, dtype: int64


>
---
Models training

In [35]:
#RF, XGB,SVM

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced' ,random_state=42, max_depth=10),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42)
}

for name, model in models.items():
    # Use the balanced training data for fitting
    model.fit(X_train_balanced, y_train_balanced)
    predictions = model.predict(X_test_scaled)

c:\Users\user\OneDrive\Desktop\Medical_AI_Suite\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [03:12:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


>
---
Evaluation

In [36]:
# Print the full report
print(classification_report(y_test, predictions, target_names=['Healthy', 'Diabetic']))

# See the raw counts
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

              precision    recall  f1-score   support

     Healthy       0.84      0.72      0.77        99
    Diabetic       0.59      0.75      0.66        55

    accuracy                           0.73       154
   macro avg       0.71      0.73      0.72       154
weighted avg       0.75      0.73      0.73       154

Confusion Matrix:
[[71 28]
 [14 41]]


>
---
Saving the model and the scaler

In [37]:

joblib.dump(model, '../models/diabetes_model.pkl')
joblib.dump(scaler, '../models/diabetes_scaler.pkl')
print("\nModel & Scaler saved successfully!")


Model & Scaler saved successfully!
